# 🏀 The Geography of Scoring: Shot Charts, Hot Zones & the 3-Point Revolution

> **Where players shoot from reveals more than how much they score.** From hex-binned efficiency maps to the animated 3-point revolution, explore the spatial dimension of basketball.

**Part 7 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Hex-Binned Shot Chart (Matplotlib)](#2-hex-binned-shot-chart)
3. [Interactive Player Shot Charts (Plotly)](#3-interactive-player-shot-charts)
4. [Hot Zone Analysis](#4-hot-zone-analysis)
5. [The 3-Point Revolution (Animated)](#5-the-3-point-revolution)
6. [Shot Type Efficiency](#6-shot-type-efficiency)
7. [Clutch Shooting](#7-clutch-shooting)
8. [Team Shot Profiles](#8-team-shot-profiles)
9. [Cleanup & Cross-Links](#9-cleanup--cross-links)

---

<a id="1-setup--data-loading"></a>
## 1. Setup & Data Loading

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "scipy>=1.12,<2"

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import duckdb
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Arc
from scipy.stats import gaussian_kde
from IPython.display import display, HTML

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import (
    COLORS, TEMPLATE, takeaway, draw_court_plotly, draw_court_matplotlib, get_connection,
)

conn = get_connection()

# --- Detect latest season ---
LATEST_SEASON = conn.sql(
    "SELECT MAX(season_year) FROM dim_game"
).fetchone()[0]
print(f"Latest season: {LATEST_SEASON}")
print("Setup complete.")

---

<a id="2-hex-binned-shot-chart"></a>
## 2. Hex-Binned Shot Chart (Matplotlib)

A hex-binned shot chart maps field goal percentage to a color gradient across the court.
Each hexagon aggregates nearby shots and shows the efficiency (FG%) in that zone,
with warmer colors indicating higher conversion rates.

In [ ]:
# Query all shots from the latest season
shots_hex = conn.sql(f"""
    SELECT sc.loc_x, sc.loc_y, sc.shot_made_flag
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    WHERE g.season_year = '{LATEST_SEASON}'
      AND sc.loc_x IS NOT NULL
      AND sc.loc_y IS NOT NULL
      AND sc.shot_made_flag IS NOT NULL
""").pl()

print(f"Shots loaded: {len(shots_hex):,} from {LATEST_SEASON}")

In [ ]:
# draw_court_matplotlib() is imported from nbadb_utils
# Draw the hex-binned chart
fig, ax = plt.subplots(figsize=(10, 9.4))

draw_court_matplotlib(ax)

hx = ax.hexbin(
    shots_hex["loc_x"].to_numpy(),
    shots_hex["loc_y"].to_numpy(),
    C=shots_hex["shot_made_flag"].to_numpy(),
    reduce_C_function=np.mean,
    gridsize=30,
    cmap="RdYlGn",
    mincnt=5,
    extent=(-250, 250, -50, 420),
    zorder=2,
    alpha=0.85,
)

cb = plt.colorbar(hx, ax=ax, label="FG%", shrink=0.8)
cb.ax.tick_params(labelsize=10)

ax.set_xlim(-260, 260)
ax.set_ylim(-55, 425)
ax.set_aspect("equal")
ax.set_title(f"League-Wide Shot Efficiency Heatmap ({LATEST_SEASON})", fontsize=16, pad=12)
ax.set_xlabel("Court X (1/10 ft)")
ax.set_ylabel("Court Y (1/10 ft)")
plt.tight_layout()
plt.show()

In [ ]:
takeaway(
    "The darkest green zones cluster around the restricted area (layups/dunks at ~65% FG) "
    "and corner threes (~38-40%). The mid-range dead zone glows yellow-red, confirming why "
    "modern NBA offenses avoid long twos in favor of threes and rim attacks."
)

---

<a id="3-interactive-player-shot-charts"></a>
## 3. Interactive Player Shot Charts (Plotly)

Use the dropdown to explore individual shot charts for the top 20 scorers.
Green circles are makes, red X's are misses. Hover for shot details.

In [ ]:
# draw_court_plotly() is imported from nbadb_utils — no local definition needed

In [ ]:
# Top 20 scorers in the latest season
top_scorers = conn.sql(f"""
    SELECT s.player_id, p.full_name, s.avg_pts
    FROM agg_player_season s
    JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
    WHERE s.season_year = '{LATEST_SEASON}'
      AND s.season_type = 'Regular Season'
      AND s.gp >= 20
    ORDER BY s.avg_pts DESC
    LIMIT 20
""").pl()

scorer_ids = top_scorers["player_id"].to_list()
scorer_names = top_scorers["full_name"].to_list()
scorer_id_str = ", ".join(str(x) for x in scorer_ids)

# Fetch shot chart data for these players
shots_player = conn.sql(f"""
    SELECT
        sc.player_id,
        p.full_name AS player_name,
        sc.loc_x,
        sc.loc_y,
        sc.shot_made_flag,
        sc.action_type,
        sc.shot_distance,
        sc.period,
        sc.zone_basic
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    JOIN dim_player p ON sc.player_id = p.player_id AND p.is_current = TRUE
    WHERE g.season_year = '{LATEST_SEASON}'
      AND sc.player_id IN ({scorer_id_str})
      AND sc.loc_x IS NOT NULL
      AND sc.loc_y IS NOT NULL
""").pl()

print(f"Loaded {len(shots_player):,} shots for {len(scorer_ids)} players")

In [ ]:
fig = go.Figure()

# Two traces per player: made + missed
for i, (pid, name) in enumerate(zip(scorer_ids, scorer_names)):
    player_shots = shots_player.filter(pl.col("player_id") == pid)

    for made, color, symbol, label in [
        (1, COLORS["positive"], "circle", "Made"),
        (0, COLORS["negative"], "x", "Missed"),
    ]:
        subset = player_shots.filter(pl.col("shot_made_flag") == made)
        fig.add_trace(go.Scatter(
            x=subset["loc_x"].to_list(),
            y=subset["loc_y"].to_list(),
            mode="markers",
            marker=dict(color=color, size=4, symbol=symbol, opacity=0.6),
            name=f"{name} ({label})",
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Action: %{customdata[1]}<br>"
                "Distance: %{customdata[2]} ft<br>"
                "Period: %{customdata[3]}<br>"
                "Zone: %{customdata[4]}<extra></extra>"
            ),
            customdata=[
                [name, row["action_type"], row["shot_distance"],
                 row["period"], row["zone_basic"]]
                for row in subset.iter_rows(named=True)
            ],
            visible=(i == 0),
        ))

# Dropdown to select player
buttons = []
for i, name in enumerate(scorer_names):
    visibility = [False] * (len(scorer_names) * 2)
    visibility[i * 2] = True
    visibility[i * 2 + 1] = True
    buttons.append(dict(
        label=name,
        method="update",
        args=[{"visible": visibility}],
    ))

fig.update_layout(
    shapes=draw_court_plotly(),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        buttons=buttons,
        bgcolor="white",
        bordercolor=COLORS["primary"],
        font=dict(size=12),
    )],
    xaxis=dict(range=[-260, 260], showgrid=False, zeroline=False, visible=False),
    yaxis=dict(range=[-60, 440], showgrid=False, zeroline=False, visible=False,
               scaleanchor="x", scaleratio=1),
    template=TEMPLATE,
    height=620, width=560,
    margin=dict(l=20, r=20, t=50, b=20),
    plot_bgcolor="white",
    title=dict(
        text=f"Player Shot Chart ({LATEST_SEASON}) -- Select Player",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    showlegend=False,
)
fig.show()

In [ ]:
takeaway(
    "Shot charts are a fingerprint. Guards spread shots along the three-point arc with "
    "pull-up midrange clusters; bigs crowd the paint and short corners; wings show "
    "the modern 'rim-and-three' pattern with nothing in between. Switch between players "
    "to see the contrast."
)

---

<a id="4-hot-zone-analysis"></a>
## 4. Hot Zone Analysis

Where does a player shoot better (or worse) than the league average? We compare
individual zone efficiency against the league baseline using a diverging colorscale.
Green zones are above average, red zones are below.

In [ ]:
# League average FG% by zone_basic
league_avg = conn.sql(f"""
    SELECT
        sc.zone_basic,
        COUNT(*) AS attempts,
        SUM(sc.shot_made_flag) AS makes,
        SUM(sc.shot_made_flag)::FLOAT / NULLIF(COUNT(*), 0) AS lg_fg_pct
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    WHERE g.season_year = '{LATEST_SEASON}'
      AND sc.zone_basic IS NOT NULL
      AND sc.shot_made_flag IS NOT NULL
    GROUP BY sc.zone_basic
    ORDER BY attempts DESC
""").pl()

print("League average FG% by zone:")
league_avg

In [ ]:
# Select 4 player archetypes: pick top scorer, a guard, a big, and a wing
# We'll use agg_shot_zones for per-player zone breakdowns
archetype_query = f"""
    WITH candidates AS (
        SELECT
            s.player_id,
            p.full_name,
            p.position,
            s.avg_pts,
            ROW_NUMBER() OVER (
                PARTITION BY
                    CASE
                        WHEN UPPER(p.position) LIKE '%G%'
                             AND UPPER(p.position) NOT LIKE '%F%' THEN 'Guard'
                        WHEN UPPER(p.position) LIKE '%C%' THEN 'Center'
                        ELSE 'Forward'
                    END
                ORDER BY s.avg_pts DESC
            ) AS pos_rank,
            CASE
                WHEN UPPER(p.position) LIKE '%G%'
                     AND UPPER(p.position) NOT LIKE '%F%' THEN 'Guard'
                WHEN UPPER(p.position) LIKE '%C%' THEN 'Center'
                ELSE 'Forward'
            END AS pos_group
        FROM agg_player_season s
        JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
        WHERE s.season_year = '{LATEST_SEASON}'
          AND s.season_type = 'Regular Season'
          AND s.gp >= 30
    )
    SELECT player_id, full_name, position, pos_group, avg_pts
    FROM candidates
    WHERE pos_rank = 1
    ORDER BY avg_pts DESC
"""

archetypes = conn.sql(archetype_query).pl()
# Also add overall #1 scorer if not already included
overall_top = conn.sql(f"""
    SELECT s.player_id, p.full_name, p.position, 'Top Scorer' AS pos_group, s.avg_pts
    FROM agg_player_season s
    JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
    WHERE s.season_year = '{LATEST_SEASON}'
      AND s.season_type = 'Regular Season'
      AND s.gp >= 30
    ORDER BY s.avg_pts DESC
    LIMIT 1
""").pl()

# Combine and deduplicate
archetype_ids = set()
archetype_list = []
for row in overall_top.iter_rows(named=True):
    if row["player_id"] not in archetype_ids:
        archetype_ids.add(row["player_id"])
        archetype_list.append(row)
for row in archetypes.iter_rows(named=True):
    if row["player_id"] not in archetype_ids:
        archetype_ids.add(row["player_id"])
        archetype_list.append(row)

# Limit to 4 players
archetype_list = archetype_list[:4]
print("Selected archetypes:")
for a in archetype_list:
    print(f"  {a['full_name']} ({a['pos_group']}) -- {a['avg_pts']:.1f} PPG")

In [ ]:
# Build league average lookup
lg_avg_map = dict(zip(
    league_avg["zone_basic"].to_list(),
    league_avg["lg_fg_pct"].to_list(),
))

# Query per-player zone data from agg_shot_zones
arch_id_str = ", ".join(str(a["player_id"]) for a in archetype_list)
player_zones = conn.sql(f"""
    SELECT
        player_id,
        zone_basic,
        SUM(attempts) AS attempts,
        SUM(makes) AS makes,
        SUM(makes)::FLOAT / NULLIF(SUM(attempts), 0) AS fg_pct
    FROM agg_shot_zones
    WHERE season_year = '{LATEST_SEASON}'
      AND player_id IN ({arch_id_str})
      AND zone_basic IS NOT NULL
    GROUP BY player_id, zone_basic
""").pl()

# Build subplots -- one treemap-style bar chart per player
n_arch = len(archetype_list)
fig = make_subplots(
    rows=1, cols=n_arch,
    subplot_titles=[a["full_name"] for a in archetype_list],
    horizontal_spacing=0.06,
)

for idx, arch in enumerate(archetype_list):
    pid = arch["player_id"]
    pz = player_zones.filter(pl.col("player_id") == pid).sort("attempts", descending=True)

    zones = pz["zone_basic"].to_list()
    fg_pcts = pz["fg_pct"].to_list()
    attempts = pz["attempts"].to_list()

    # Compute differential vs league avg
    diffs = [
        (fg - lg_avg_map.get(z, 0.45)) * 100
        for z, fg in zip(zones, fg_pcts)
    ]

    # Color by diff: green > 0, red < 0
    bar_colors = [
        COLORS["positive"] if d > 0 else COLORS["negative"]
        for d in diffs
    ]

    fig.add_trace(
        go.Bar(
            y=zones[::-1],
            x=diffs[::-1],
            orientation="h",
            marker_color=bar_colors[::-1],
            text=[
                f"{d:+.1f}% ({a} att)"
                for d, a in zip(diffs[::-1], attempts[::-1])
            ],
            textposition="outside",
            hovertemplate=(
                "%{y}<br>"
                "FG%: %{customdata[0]:.1f}%<br>"
                "League Avg: %{customdata[1]:.1f}%<br>"
                "Diff: %{x:+.1f}%<extra></extra>"
            ),
            customdata=[
                [fg * 100, lg_avg_map.get(z, 0.45) * 100]
                for z, fg in zip(zones[::-1], fg_pcts[::-1])
            ],
            showlegend=False,
        ),
        row=1, col=idx + 1,
    )
    fig.update_xaxes(zeroline=True, zerolinecolor="gray", zerolinewidth=1,
                      row=1, col=idx + 1)

fig.update_layout(
    template=TEMPLATE,
    height=450,
    width=1100,
    title=dict(
        text=f"Hot Zone Analysis: FG% vs League Average ({LATEST_SEASON})",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=20, r=20, t=80, b=20),
)
fig.show()

In [ ]:
takeaway(
    "Player archetypes show distinct hot zone signatures. Guards often exceed league "
    "average from above the break and mid-range; centers dominate the restricted area "
    "but struggle beyond 15 feet. The best scorers tend to be above average across "
    "multiple zones -- true shot creators have no dead spots."
)

---

<a id="5-the-3-point-revolution"></a>
## 5. The 3-Point Revolution (Animated)

The most dramatic trend in modern basketball: the mid-range shot is disappearing
as teams optimize for threes and rim attacks. We animate shot density across
seasons to watch the revolution unfold.

In [ ]:
# Get shot locations by season, sampling for performance
revolution_data = conn.sql("""
    SELECT
        g.season_year,
        sc.loc_x,
        sc.loc_y,
        sc.shot_made_flag,
        sc.zone_basic
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    WHERE sc.loc_x IS NOT NULL
      AND sc.loc_y IS NOT NULL
      AND sc.shot_made_flag IS NOT NULL
      AND sc.zone_basic IS NOT NULL
    ORDER BY g.season_year
""").pl()

all_seasons = sorted(revolution_data["season_year"].unique().to_list())
print(f"Seasons available: {all_seasons[0]} to {all_seasons[-1]} ({len(all_seasons)} seasons)")
print(f"Total shots: {len(revolution_data):,}")

In [ ]:
# Compute shot distribution by zone per season
zone_trend = (
    revolution_data
    .group_by(["season_year", "zone_basic"])
    .agg([
        pl.len().alias("shot_count"),
        pl.col("shot_made_flag").mean().alias("fg_pct"),
    ])
    .with_columns(
        (pl.col("shot_count") / pl.col("shot_count").sum().over("season_year") * 100)
        .alias("pct_of_shots")
    )
    .sort(["season_year", "zone_basic"])
)

# Identify key zones for trending
key_zones = [
    "Restricted Area",
    "In The Paint (Non-RA)",
    "Mid-Range",
    "Above the Break 3",
    "Left Corner 3",
    "Right Corner 3",
]

zone_colors = {
    "Restricted Area": COLORS["positive"],
    "In The Paint (Non-RA)": COLORS["accent"],
    "Mid-Range": COLORS["negative"],
    "Above the Break 3": COLORS["primary"],
    "Left Corner 3": "#552583",
    "Right Corner 3": "#CE1141",
}

In [ ]:
# Animated shot density on court using 2D histogram per season
# Build frames for animation

# Use a subsample per season for KDE performance
SAMPLE_PER_SEASON = 5000

# Create base figure with first season
first_season = all_seasons[0]
first_data = revolution_data.filter(pl.col("season_year") == first_season)
if len(first_data) > SAMPLE_PER_SEASON:
    first_data = first_data.sample(SAMPLE_PER_SEASON, seed=42)

fig_anim = go.Figure(
    data=[go.Histogram2dContour(
        x=first_data["loc_x"].to_list(),
        y=first_data["loc_y"].to_list(),
        colorscale="YlOrRd",
        ncontours=15,
        showscale=True,
        colorbar=dict(title="Shot Density"),
        contours=dict(showlines=False),
    )]
)

# Build animation frames
frames = []
for season in all_seasons:
    season_data = revolution_data.filter(pl.col("season_year") == season)
    if len(season_data) > SAMPLE_PER_SEASON:
        season_data = season_data.sample(SAMPLE_PER_SEASON, seed=42)

    frames.append(go.Frame(
        data=[go.Histogram2dContour(
            x=season_data["loc_x"].to_list(),
            y=season_data["loc_y"].to_list(),
            colorscale="YlOrRd",
            ncontours=15,
            showscale=True,
            colorbar=dict(title="Shot Density"),
            contours=dict(showlines=False),
        )],
        name=str(season),
        layout=go.Layout(
            title=dict(
                text=f"Shot Density Evolution -- {season}",
                font=dict(size=16, color=COLORS["primary"]),
            ),
        ),
    ))

fig_anim.frames = frames

# Animation controls
fig_anim.update_layout(
    shapes=draw_court_plotly(line_color="#333333", line_width=1.5),
    xaxis=dict(range=[-260, 260], showgrid=False, zeroline=False, visible=False),
    yaxis=dict(range=[-60, 440], showgrid=False, zeroline=False, visible=False,
               scaleanchor="x", scaleratio=1),
    template=TEMPLATE,
    height=650, width=600,
    margin=dict(l=20, r=20, t=60, b=80),
    plot_bgcolor="white",
    title=dict(
        text=f"Shot Density Evolution -- {first_season}",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.05, y=-0.04,
        xanchor="left", yanchor="top",
        buttons=[
            dict(label="Play",
                 method="animate",
                 args=[None, dict(
                     frame=dict(duration=600, redraw=True),
                     fromcurrent=True,
                     transition=dict(duration=300),
                 )]),
            dict(label="Pause",
                 method="animate",
                 args=[[None], dict(
                     frame=dict(duration=0, redraw=False),
                     mode="immediate",
                     transition=dict(duration=0),
                 )]),
        ],
    )],
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix="Season: ", font=dict(size=14)),
        steps=[
            dict(args=[
                [str(season)],
                dict(frame=dict(duration=600, redraw=True),
                     mode="immediate",
                     transition=dict(duration=300)),
            ], label=str(season), method="animate")
            for season in all_seasons
        ],
        x=0.05, len=0.9,
        y=-0.02,
        xanchor="left",
        transition=dict(duration=300),
    )],
)

fig_anim.show()

In [ ]:
# Trend line chart: % of shots by zone over time
fig_trend = go.Figure()

for zone in key_zones:
    zdata = zone_trend.filter(pl.col("zone_basic") == zone)
    if zdata.is_empty():
        continue
    fig_trend.add_trace(go.Scatter(
        x=zdata["season_year"].to_list(),
        y=zdata["pct_of_shots"].to_list(),
        mode="lines+markers",
        name=zone,
        line=dict(color=zone_colors.get(zone, COLORS["neutral"]), width=2.5),
        marker=dict(size=5),
        hovertemplate=f"{zone}<br>Season: %{{x}}<br>%{{y:.1f}}% of shots<extra></extra>",
    ))

fig_trend.update_layout(
    template=TEMPLATE,
    title=dict(
        text="The 3-Point Revolution: Shot Distribution by Zone Over Time",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Season",
    yaxis_title="% of Total Shots",
    height=500,
    width=1000,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_trend.show()

In [ ]:
# Compute the magnitude of the shift
earliest = zone_trend.filter(pl.col("season_year") == all_seasons[0])
latest = zone_trend.filter(pl.col("season_year") == all_seasons[-1])

# 3pt share change
three_zones = ["Above the Break 3", "Left Corner 3", "Right Corner 3"]
early_3pt = earliest.filter(pl.col("zone_basic").is_in(three_zones))["pct_of_shots"].sum()
late_3pt = latest.filter(pl.col("zone_basic").is_in(three_zones))["pct_of_shots"].sum()

# Mid-range change
early_mid = earliest.filter(pl.col("zone_basic") == "Mid-Range")["pct_of_shots"].sum()
late_mid = latest.filter(pl.col("zone_basic") == "Mid-Range")["pct_of_shots"].sum()

takeaway(
    f"Three-point attempts grew from {early_3pt:.1f}% to {late_3pt:.1f}% of all shots "
    f"({all_seasons[0]} to {all_seasons[-1]}), while mid-range shots collapsed from "
    f"{early_mid:.1f}% to {late_mid:.1f}%. The animated density map shows the mid-range "
    "literally disappearing as shot density migrates to the arc and the rim -- the "
    "spatial signature of the analytics revolution."
)

---

<a id="6-shot-type-efficiency"></a>
## 6. Shot Type Efficiency

Not all shots are created equal. We rank the most common shot types (action_type)
by their field goal percentage and volume.

In [ ]:
# Shot type efficiency across all available data
shot_types = conn.sql(f"""
    SELECT
        sc.action_type,
        COUNT(*) AS attempts,
        SUM(sc.shot_made_flag) AS makes,
        SUM(sc.shot_made_flag)::FLOAT / NULLIF(COUNT(*), 0) AS fg_pct
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    WHERE g.season_year = '{LATEST_SEASON}'
      AND sc.action_type IS NOT NULL
      AND sc.shot_made_flag IS NOT NULL
    GROUP BY sc.action_type
    HAVING COUNT(*) >= 500
    ORDER BY fg_pct DESC
""").pl()

print(f"Shot types with >= 500 attempts: {len(shot_types)}")
shot_types

In [ ]:
# Bar chart colored by efficiency tier
action_types = shot_types["action_type"].to_list()
fg_pcts = shot_types["fg_pct"].to_list()
attempts = shot_types["attempts"].to_list()

bar_colors = [
    COLORS["positive"] if fg > 0.50
    else COLORS["accent"] if fg >= 0.40
    else COLORS["negative"]
    for fg in fg_pcts
]

fig_st = go.Figure()
fig_st.add_trace(go.Bar(
    y=action_types[::-1],
    x=[fg * 100 for fg in fg_pcts[::-1]],
    orientation="h",
    marker_color=bar_colors[::-1],
    text=[
        f"{fg:.1%} ({att:,} att)"
        for fg, att in zip(fg_pcts[::-1], attempts[::-1])
    ],
    textposition="outside",
    hovertemplate="%{y}<br>FG%: %{x:.1f}%<extra></extra>",
))

# Reference lines
fig_st.add_vline(x=50, line_dash="dash", line_color=COLORS["positive"],
                  annotation_text="50% FG", annotation_position="top")
fig_st.add_vline(x=40, line_dash="dash", line_color=COLORS["accent"],
                  annotation_text="40% FG", annotation_position="top")

fig_st.update_layout(
    template=TEMPLATE,
    title=dict(
        text=f"Shot Type Efficiency ({LATEST_SEASON}, min 500 attempts)",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Field Goal %",
    height=max(400, len(action_types) * 35 + 100),
    width=900,
    margin=dict(l=200, r=100, t=60, b=40),
    showlegend=False,
)
fig_st.show()

In [ ]:
takeaway(
    "Dunks and layups convert above 60%, while contested jump shots hover around 35-40%. "
    "The efficiency gap explains modern shot selection: teams maximize attempts at the rim "
    "(highest FG%) and from three (highest expected value per shot) while minimizing "
    "long twos and contested mid-range pull-ups."
)

---

<a id="7-clutch-shooting"></a>
## 7. Clutch Shooting

Clutch time: period >= 4, 2 minutes or fewer remaining, game within 5 points.
How does shooting efficiency change under pressure? We compare FG% by zone in
clutch vs non-clutch situations.

In [ ]:
# Clutch vs non-clutch FG% by zone
clutch_data = conn.sql(f"""
    WITH labeled AS (
        SELECT
            sc.zone_basic,
            sc.shot_made_flag,
            CASE
                WHEN sc.period >= 4 AND sc.minutes_remaining <= 2
                THEN 'Clutch'
                ELSE 'Non-Clutch'
            END AS situation
        FROM fact_shot_chart sc
        JOIN dim_game g ON sc.game_id = g.game_id
        WHERE g.season_year = '{LATEST_SEASON}'
          AND sc.zone_basic IS NOT NULL
          AND sc.shot_made_flag IS NOT NULL
    )
    SELECT
        zone_basic,
        situation,
        COUNT(*) AS attempts,
        SUM(shot_made_flag) AS makes,
        SUM(shot_made_flag)::FLOAT / NULLIF(COUNT(*), 0) AS fg_pct
    FROM labeled
    GROUP BY zone_basic, situation
    HAVING COUNT(*) >= 20
    ORDER BY zone_basic, situation
""").pl()

print(f"Clutch data rows: {len(clutch_data)}")
clutch_data

In [ ]:
# Grouped bar chart: clutch vs non-clutch per zone
zones_with_both = (
    clutch_data
    .group_by("zone_basic")
    .agg(pl.col("situation").n_unique().alias("n_situations"))
    .filter(pl.col("n_situations") == 2)["zone_basic"]
    .to_list()
)

fig_clutch = go.Figure()

for situation, color in [
    ("Non-Clutch", COLORS["primary"]),
    ("Clutch", COLORS["secondary"]),
]:
    sit_data = clutch_data.filter(
        (pl.col("situation") == situation) & pl.col("zone_basic").is_in(zones_with_both)
    ).sort("zone_basic")

    fig_clutch.add_trace(go.Bar(
        x=sit_data["zone_basic"].to_list(),
        y=[fg * 100 for fg in sit_data["fg_pct"].to_list()],
        name=situation,
        marker_color=color,
        text=[
            f"{fg:.1%}<br>({att})"
            for fg, att in zip(sit_data["fg_pct"].to_list(), sit_data["attempts"].to_list())
        ],
        textposition="outside",
        hovertemplate="%{x}<br>%{y:.1f}% FG<extra></extra>",
    ))

fig_clutch.update_layout(
    template=TEMPLATE,
    barmode="group",
    title=dict(
        text=f"Clutch vs Non-Clutch FG% by Zone ({LATEST_SEASON})",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Shot Zone",
    yaxis_title="Field Goal %",
    yaxis_range=[0, 80],
    height=500,
    width=950,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_clutch.show()

In [ ]:
# Compute the biggest clutch drop-off
clutch_pivot = (
    clutch_data
    .filter(pl.col("zone_basic").is_in(zones_with_both))
    .pivot(on="situation", index="zone_basic", values="fg_pct")
)

if "Clutch" in clutch_pivot.columns and "Non-Clutch" in clutch_pivot.columns:
    clutch_pivot = clutch_pivot.with_columns(
        ((pl.col("Clutch") - pl.col("Non-Clutch")) * 100).alias("diff_pct")
    ).sort("diff_pct")

    worst_zone = clutch_pivot[0, "zone_basic"]
    worst_diff = clutch_pivot[0, "diff_pct"]

    takeaway(
        f"Shooting efficiency drops across nearly every zone in clutch situations. "
        f"The biggest decline is in the {worst_zone} ({worst_diff:+.1f} percentage points). "
        "Restricted area shots tend to hold up best under pressure -- rim attacks remain "
        "the most reliable clutch strategy. Three-pointers show the most variance, "
        "reflecting the higher difficulty of perimeter shooting under defensive intensity."
    )
else:
    takeaway(
        "Shooting efficiency generally declines in clutch situations, with rim attacks "
        "maintaining the most stability under pressure."
    )

---

<a id="8-team-shot-profiles"></a>
## 8. Team Shot Profiles

How does each team distribute its shots across court zones? We compute the
percentage of each team's shots from each zone and rank by an "analytics score"
-- the share of shots from the three most efficient zones (restricted area,
above-the-break 3, and corner 3).

In [ ]:
# Team shot distribution by zone
team_zones = conn.sql(f"""
    SELECT
        t.abbreviation,
        sc.zone_basic,
        COUNT(*) AS shots,
        SUM(sc.shot_made_flag)::FLOAT / NULLIF(COUNT(*), 0) AS fg_pct
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    JOIN dim_team t ON sc.team_id = t.team_id
    WHERE g.season_year = '{LATEST_SEASON}'
      AND sc.zone_basic IS NOT NULL
      AND sc.shot_made_flag IS NOT NULL
    GROUP BY t.abbreviation, sc.zone_basic
    ORDER BY t.abbreviation, sc.zone_basic
""").pl()

# Compute shot share per team
team_zones = team_zones.with_columns(
    (pl.col("shots") / pl.col("shots").sum().over("abbreviation") * 100)
    .alias("pct_of_shots")
)

print(f"Teams: {team_zones['abbreviation'].n_unique()}")
print(f"Zones: {sorted(team_zones['zone_basic'].unique().to_list())}")

In [ ]:
# Compute analytics score per team
analytics_zones = ["Restricted Area", "Above the Break 3", "Left Corner 3", "Right Corner 3"]
analytics_score = (
    team_zones
    .filter(pl.col("zone_basic").is_in(analytics_zones))
    .group_by("abbreviation")
    .agg(pl.col("pct_of_shots").sum().alias("analytics_score"))
    .sort("analytics_score", descending=True)
)

team_order = analytics_score["abbreviation"].to_list()

# Stacked horizontal bar chart
all_zones = sorted(team_zones["zone_basic"].unique().to_list())

zone_display_colors = {
    "Restricted Area": COLORS["positive"],
    "In The Paint (Non-RA)": COLORS["accent"],
    "Mid-Range": COLORS["negative"],
    "Above the Break 3": COLORS["primary"],
    "Left Corner 3": "#552583",
    "Right Corner 3": "#CE1141",
    "Backcourt": COLORS["neutral"],
}

fig_team = go.Figure()

for zone in all_zones:
    zone_data = team_zones.filter(pl.col("zone_basic") == zone)
    # Map to team order
    zone_map = dict(zip(zone_data["abbreviation"].to_list(),
                         zone_data["pct_of_shots"].to_list()))

    fig_team.add_trace(go.Bar(
        y=team_order[::-1],
        x=[zone_map.get(t, 0) for t in team_order[::-1]],
        name=zone,
        orientation="h",
        marker_color=zone_display_colors.get(zone, COLORS["neutral"]),
        hovertemplate=f"{zone}<br>%{{y}}: %{{x:.1f}}%<extra></extra>",
    ))

fig_team.update_layout(
    template=TEMPLATE,
    barmode="stack",
    title=dict(
        text=f"Team Shot Distribution by Zone ({LATEST_SEASON}) -- Sorted by Analytics Score",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="% of Total Shots",
    height=max(500, len(team_order) * 22 + 100),
    width=1000,
    margin=dict(l=60, r=20, t=60, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_team.show()

In [ ]:
# Identify most and least analytically optimized teams
most_analytic = analytics_score.head(3)
least_analytic = analytics_score.tail(3)

most_names = ", ".join(most_analytic["abbreviation"].to_list())
most_scores = ", ".join(f"{s:.1f}%" for s in most_analytic["analytics_score"].to_list())
least_names = ", ".join(least_analytic["abbreviation"].to_list())
least_scores = ", ".join(f"{s:.1f}%" for s in least_analytic["analytics_score"].to_list())

takeaway(
    f"Most analytically optimized teams: <strong>{most_names}</strong> ({most_scores} "
    f"of shots from rim + 3). Least optimized: <strong>{least_names}</strong> "
    f"({least_scores}). The spread between top and bottom teams can exceed 15 percentage "
    "points, reflecting fundamentally different offensive philosophies. Teams at the "
    "top maximize expected value per shot; teams at the bottom retain more mid-range "
    "and paint-non-RA volume."
)

---

<a id="9-cleanup--cross-links"></a>
## 9. Cleanup & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### Summary

This notebook explored the spatial dimension of NBA scoring:

1. **Hex-binned heatmaps** confirmed that rim shots and corner threes are the most efficient zones
2. **Interactive player shot charts** revealed distinct shooting fingerprints per player
3. **Hot zone analysis** showed how player archetypes differ from league averages
4. **The 3-point revolution** animated the dramatic migration from mid-range to arc shots
5. **Shot type efficiency** ranked action types from dunks (65%+) to contested jumpers (~35%)
6. **Clutch shooting** showed efficiency declines under pressure, especially from three
7. **Team shot profiles** ranked franchises by analytics optimization

---

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| **Part 7** | **Shot Chart Analysis** (this notebook) | **Spatial Shot Analysis & 3-Point Revolution** |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)